In [1]:
import pandas as pd
df_ground_truth = pd.read_csv('outputs/DE_12k_IR_0.007_0HP_1HP_2HP_3HP_120min/ground_truth.csv')
df_ground_truth.head()

,t_start,t_end,label,transition,health_state,fault_size
0,0.0,0.5,Normal_0.csv,False,healthy,—
1,1.5,2.0,Normal_0.csv,False,healthy,—
2,3.0,3.5,Normal_0.csv,False,healthy,—
3,4.5,5.0,Normal_0.csv,False,healthy,—
4,6.0,6.5,Normal_0.csv,False,healthy,—


In [2]:
# Create another column named 'Concerned Sliced CSV'
df_ground_truth['Concerned Sliced CSV'] = pd.NA
# Merge the t_start and t_end columns into a new column named 'Concerned Sliced CSV' with the format 't_start_t_end.csv'
df_ground_truth['Concerned Sliced CSV'] = df_ground_truth['t_start'].astype(str) + '_' + df_ground_truth['t_end'].astype(str) + '.csv'
df_ground_truth.head()

,t_start,t_end,label,transition,health_state,fault_size,Concerned Sliced CSV
0,0.0,0.5,Normal_0.csv,False,healthy,—,0.0_0.5.csv
1,1.5,2.0,Normal_0.csv,False,healthy,—,1.5_2.0.csv
2,3.0,3.5,Normal_0.csv,False,healthy,—,3.0_3.5.csv
3,4.5,5.0,Normal_0.csv,False,healthy,—,4.5_5.0.csv
4,6.0,6.5,Normal_0.csv,False,healthy,—,6.0_6.5.csv


In [3]:
df_ground_truth.label.unique()

array(['Normal_0.csv', 'Normal_1.csv', 'Normal_2.csv', 'Normal_3.csv',
       '12k_DE_IR007_2.csv', '12k_DE_IR007_3.csv', '12k_DE_IR007_0.csv',
       '12k_DE_IR007_1.csv'], dtype=object)

In [4]:
# Assign a label to each unique existing label value, starting from 'S1', 'S2', 'S3', and so on
label_map = {old_label: f"S{i}" for i, old_label in enumerate(df_ground_truth['label'].unique(), start=1)}
df_ground_truth['Ground Truth State'] = df_ground_truth['label'].map(label_map)
df_ground_truth

,t_start,t_end,label,transition,health_state,fault_size,Concerned Sliced CSV,Ground Truth State
0,0.0,0.5,Normal_0.csv,False,healthy,—,0.0_0.5.csv,S1
1,1.5,2.0,Normal_0.csv,False,healthy,—,1.5_2.0.csv,S1
2,3.0,3.5,Normal_0.csv,False,healthy,—,3.0_3.5.csv,S1
3,4.5,5.0,Normal_0.csv,False,healthy,—,4.5_5.0.csv,S1
4,6.0,6.5,Normal_0.csv,False,healthy,—,6.0_6.5.csv,S1
...,...,...,...,...,...,...,...,...
4795,7192.5,7193.0,12k_DE_IR007_1.csv,False,faulty,0.007,7192.5_7193.0.csv,S8
4796,7194.0,7194.5,12k_DE_IR007_2.csv,False,faulty,0.007,7194.0_7194.5.csv,S5
4797,7195.5,7196.0,12k_DE_IR007_2.csv,False,faulty,0.007,7195.5_7196.0.csv,S5
4798,7197.0,7197.5,12k_DE_IR007_2.csv,False,faulty,0.007,7197.0_7197.5.csv,S5


In [5]:
from pathlib import Path

output_dir = Path('outputs/DE_12k_IR_0.007_0HP_1HP_2HP_3HP_120min')
updated_files = []
missing_files = []

for _, row in df_ground_truth.drop_duplicates(subset=['Concerned Sliced CSV']).iterrows():
    csv_path = output_dir / row['Concerned Sliced CSV']
    if not csv_path.exists():
        missing_files.append(csv_path.name)
        continue

    df_slice = pd.read_csv(csv_path)
    df_slice['Ground Truth State'] = row['Ground Truth State']
    df_slice.to_csv(csv_path, index=False)
    updated_files.append(csv_path.name)

print(f'Updated {len(updated_files)} sliced CSV files.')
if missing_files:
    print(f'Missing {len(missing_files)} files: {missing_files[:10]}')

df_ground_truth[['Concerned Sliced CSV', 'Ground Truth State']].head()

Updated 4800 sliced CSV files.


,Concerned Sliced CSV,Ground Truth State
0,0.0_0.5.csv,S1
1,1.5_2.0.csv,S1
2,3.0_3.5.csv,S1
3,4.5_5.0.csv,S1
4,6.0_6.5.csv,S1


In [6]:
df_ground_truth.to_csv(output_dir / 'ground_truth_with_states.csv', index=False)